# Real Examples: Parsing, Chunking, Retrieval

Notebook de transparence sur le clean-room BOFIP-only.

Portée volontaire:
- parsing réel BOFIP
- chunking réel
- retrieval réel
- pas de génération LLM ici

Point important:
- les exemples très faibles du type `BOI-ANNX-*`, `BOI-FORM-*`, `Zone 1 :`, `Numéro SIREN :` proviennent d'anciens essais mixtes ou de stratégies rejetées
- le baseline actuel retenu pour avancer est:
  - `Commentaire` only
  - `section_window`


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from bofip_cleanroom.jsonio import read_jsonl
from bofip_cleanroom.models import chunk_node_from_dict, raw_document_from_dict

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def load_jsonl(path):
    return read_jsonl(Path(path))

raw_docs_50 = [raw_document_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "raw_docs_sample_50.jsonl")]
raw_docs_200 = [raw_document_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "raw_docs_sample_200.jsonl")]
raw_docs_1000 = [raw_document_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "raw_docs_sample_1000.jsonl")]

chunks_10_pp = [chunk_node_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "chunks_paragraph_preserving_sample_10.jsonl")]
chunks_10_sw = [chunk_node_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "chunks_section_window_sample_10.jsonl")]
chunks_50_sw = [chunk_node_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "chunks_section_window_sample_50.jsonl")]
chunks_200_sw = [chunk_node_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "chunks_section_window_sample_200.jsonl")]
chunks_1000_sw = [chunk_node_from_dict(item) for item in load_jsonl(PROJECT_ROOT / "data" / "interim" / "chunks_section_window_sample_1000.jsonl")]

chunk_by_id_200 = {chunk.chunk_id: chunk for chunk in chunks_200_sw}
chunk_by_id_1000 = {chunk.chunk_id: chunk for chunk in chunks_1000_sw}

reports = {
    "parse_full": load_json(PROJECT_ROOT / "data" / "reports" / "phase0_full_parse_audit.json"),
    "sample50_summary": load_json(PROJECT_ROOT / "data" / "reports" / "phase1_extract_summary_sample_50.json"),
    "sample200_summary": load_json(PROJECT_ROOT / "data" / "reports" / "phase1_extract_summary_sample_200.json"),
    "sample1000_summary": load_json(PROJECT_ROOT / "data" / "reports" / "phase1_extract_summary_sample_1000.json"),
    "chunk50": load_json(PROJECT_ROOT / "data" / "reports" / "phase2_chunks_section_window_sample_50.json"),
    "chunk200": load_json(PROJECT_ROOT / "data" / "reports" / "phase2_chunks_section_window_sample_200.json"),
    "chunk1000": load_json(PROJECT_ROOT / "data" / "reports" / "phase2_chunks_section_window_sample_1000.json"),
    "lex200": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_batch_eval_chunks_section_window_sample_200.json"),
    "lex1000": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_batch_eval_chunks_section_window_sample_1000.json"),
    "dense50": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_dense_eval_chunks_section_window_sample_50_intfloat__multilingual-e5-base.json"),
    "dense200": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_dense_eval_chunks_section_window_sample_200_intfloat__multilingual-e5-base.json"),
    "dense1000": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_dense_eval_chunks_section_window_sample_1000_intfloat__multilingual-e5-base.json"),
    "hyb200_w": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_hybrid_eval_chunks_section_window_sample_200_intfloat__multilingual-e5-base_lw2p0_dw1p0.json"),
    "hyb1000_w": load_json(PROJECT_ROOT / "data" / "reports" / "phase3_hybrid_eval_chunks_section_window_sample_1000_intfloat__multilingual-e5-base_lw2p0_dw1p0.json"),
}

print("Loaded clean-room artifacts.")


Loaded clean-room artifacts.


In [2]:
{
    "content_docs_total": reports["parse_full"]["documents_considered"],
    "parse_failures": reports["parse_full"]["failure_count"],
    "sample_50_docs": reports["sample50_summary"]["sample_size"],
    "sample_200_docs": reports["sample200_summary"]["sample_size"],
    "sample_1000_docs": reports["sample1000_summary"]["sample_size"],
    "chunks_50_section_window": reports["chunk50"]["chunk_count"],
    "chunks_200_section_window": reports["chunk200"]["chunk_count"],
    "chunks_1000_section_window": reports["chunk1000"]["chunk_count"],
    "lexical_200": reports["lex200"]["metrics"],
    "dense_200": reports["dense200"]["metrics"],
    "hybrid_weighted_200": reports["hyb200_w"]["metrics"],
    "lexical_1000": reports["lex1000"]["metrics"],
    "dense_1000": reports["dense1000"]["metrics"],
    "hybrid_weighted_1000": reports["hyb1000_w"]["metrics"],
}


{'content_docs_total': 6295,
 'parse_failures': 0,
 'sample_50_docs': 50,
 'sample_200_docs': 200,
 'sample_1000_docs': 1000,
 'chunks_50_section_window': 758,
 'chunks_200_section_window': 2716,
 'chunks_1000_section_window': 11931,
 'lexical_200': {'hit@1': 0.9, 'hit@3': 1.0, 'hit@5': 1.0},
 'dense_200': {'hit@1': 0.9, 'hit@3': 0.9, 'hit@5': 1.0},
 'hybrid_weighted_200': {'hit@1': 0.95, 'hit@3': 1.0, 'hit@5': 1.0},
 'lexical_1000': {'hit@1': 0.85, 'hit@3': 1.0, 'hit@5': 1.0},
 'dense_1000': {'hit@1': 0.65, 'hit@3': 0.85, 'hit@5': 0.9},
 'hybrid_weighted_1000': {'hit@1': 0.85, 'hit@3': 0.95, 'hit@5': 0.95}}

## 1. Pourquoi certains petits chunks vus avant étaient mauvais

La question n'est pas "est-ce un chunk ?" mais "est-ce un chunk acceptable pour le baseline ?".

Réponse:
- oui, les extraits très courts étaient bien des chunks
- non, ils n'étaient pas acceptables comme baseline
- c'est précisément pour ça que `paragraph_preserving` n'a pas été retenu
- et pour ça aussi que le baseline actuel ne mélange pas tout BOFIP indistinctement


In [3]:
def shortest_chunks(chunks, n=12):
    rows = []
    for chunk in sorted(chunks, key=lambda c: (c.token_count, len(c.text), c.boi_reference))[:n]:
        rows.append({
            "tokens": chunk.token_count,
            "boi_reference": chunk.boi_reference,
            "chunk_kind": chunk.chunk_kind,
            "text": chunk.text[:160],
        })
    return rows

{
    "old_sample10_paragraph_preserving_shortest": shortest_chunks(chunks_10_pp, 12),
    "old_sample10_section_window_shortest": shortest_chunks(chunks_10_sw, 12),
    "current_sample1000_section_window_shortest": shortest_chunks(chunks_1000_sw, 12),
}


{'old_sample10_paragraph_preserving_shortest': [{'tokens': 1,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': 'Département.'},
  {'tokens': 2,
   'boi_reference': 'BOI-BAREME-000024-20160706',
   'chunk_kind': 'paragraph',
   'text': 'Remarques :'},
  {'tokens': 4,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': 'Zone 1 :'},
  {'tokens': 4,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': 'Zone 1 :'},
  {'tokens': 4,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': 'Zone 2 :'},
  {'tokens': 4,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': '-Zone 1 :'},
  {'tokens': 4,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': '-Zone 2 :'},
  {'tokens': 4,
   'boi_reference': 'BOI-ANNX-000098-20120912',
   'chunk_kind': 'paragraph',
   'text': '- Zone2 :'}

## 2. Quels documents réels ont été pris dans le subset `50` commentaires


In [4]:
[
    {
        "boi_reference": doc.boi_reference,
        "title": doc.title,
        "sections": len(doc.sections),
        "paragraphs": len(doc.paragraphs),
        "tables": len(doc.tables),
        "legal_refs": len(doc.legal_refs),
    }
    for doc in raw_docs_50
][:20]


[{'boi_reference': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
  'title': "BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux",
  'sections': 11,
  'paragraphs': 131,
  'tables': 2,
  'legal_refs': 22},
 {'boi_reference': 'BOI-IF-TFNB-10-40-60-20120912',
  'title': "IF -Taxe foncière sur les propriétés non bâties - Champ d'application et territorialité – Exonérations permanentes - Terrains plantés en oliviers",
  'sections': 9,
  'paragraphs': 21,
  'tables': 0,
  'legal_refs': 3},
 {'boi_reference': 'BOI-PAT-IFI-60-30-20190502',
  'title': 'PAT - IFI - Contrôle, pénalités et contentieux - Contentieux',
  'sections': 1,
  'paragraphs': 21,
  'tables': 0,
  'legal_refs': 1},
 {'boi_reference': 'BOI-INT-CVB-DEU-10-40-20120912',
  'title': "INT - Convention fiscale entre la France et l'Allemagne

In [5]:
def get_doc(raw_docs, boi_reference):
    for doc in raw_docs:
        if doc.boi_reference == boi_reference:
            return doc
    raise KeyError(boi_reference)

def doc_overview(raw_docs, boi_reference, section_limit=12, paragraph_limit=12, table_limit=3):
    doc = get_doc(raw_docs, boi_reference)
    return {
        "boi_reference": doc.boi_reference,
        "title": doc.title,
        "content_type": doc.content_type,
        "publication_date": doc.publication_date,
        "source_url": doc.source_url,
        "section_count": len(doc.sections),
        "paragraph_count": len(doc.paragraphs),
        "table_count": len(doc.tables),
        "sections_preview": [
            {
                "level": section.level,
                "title": section.title,
                "path": section.path,
            }
            for section in doc.sections[:section_limit]
        ],
        "paragraphs_preview": [
            {
                "order_index": p.order_index,
                "section_id": p.section_id,
                "paragraph_number": p.paragraph_number,
                "text": p.text[:240],
            }
            for p in doc.paragraphs[:paragraph_limit]
        ],
        "tables_preview": [
            {
                "caption": table.caption,
                "headers": table.headers,
                "rows": table.rows[:5],
                "linearized_text": table.linearized_text[:400],
            }
            for table in doc.tables[:table_limit]
        ],
    }

doc_overview(raw_docs_50, "BOI-BIC-CHAMP-80-20-20-20-20240703")


{'boi_reference': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
 'title': "BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux",
 'content_type': 'Commentaire',
 'publication_date': '2024-07-03',
 'source_url': 'https://bofip.impots.gouv.fr/bofip/5357-PGP.html/identifiant=BOI-BIC-CHAMP-80-20-20-20-20240703',
 'section_count': 11,
 'paragraph_count': 131,
 'table_count': 2,
 'sections_preview': [{'level': 0,
   'title': "BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux",
   'path': ["BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeu

In [6]:
doc_overview(raw_docs_50, "BOI-IS-GPE-50-10-30-20210811")


{'boi_reference': 'BOI-IS-GPE-50-10-30-20210811',
 'title': "IS - Régime fiscal des groupes de sociétés - Opérations de restructurations du groupe - Absorption de la société mère, de l'entité mère non résidente ou d'une société étrangère - Sort des déficits, charges financières nettes non déduites et capacité de déduction inemployée de l'ancien groupe",
 'content_type': 'Commentaire',
 'publication_date': '2021-08-11',
 'source_url': 'https://bofip.impots.gouv.fr/bofip/6028-PGP.html/identifiant=BOI-IS-GPE-50-10-30-20210811',
 'section_count': 19,
 'paragraph_count': 368,
 'table_count': 13,
 'sections_preview': [{'level': 0,
   'title': "IS - Régime fiscal des groupes de sociétés - Opérations de restructurations du groupe - Absorption de la société mère, de l'entité mère non résidente ou d'une société étrangère - Sort des déficits, charges financières nettes non déduites et capacité de déduction inemployée de l'ancien groupe",
   'path': ["IS - Régime fiscal des groupes de sociétés - O

In [7]:
doc_overview(raw_docs_50, "BOI-RES-TVA-000074-20210309")


{'boi_reference': 'BOI-RES-TVA-000074-20210309',
 'title': 'RES - Taxe sur la valeur ajoutée - Liquidation - Taux de TVA applicable aux systèmes de fixation permettant d’accrocher un fauteuil roulant à une trottinette électrique',
 'content_type': 'Commentaire',
 'publication_date': '2021-03-09',
 'source_url': 'https://bofip.impots.gouv.fr/bofip/12471-PGP.html/identifiant=BOI-RES-TVA-000074-20210309',
 'section_count': 1,
 'paragraph_count': 9,
 'table_count': 0,
 'sections_preview': [{'level': 0,
   'title': 'RES - Taxe sur la valeur ajoutée - Liquidation - Taux de TVA applicable aux systèmes de fixation permettant d’accrocher un fauteuil roulant à une trottinette électrique',
   'path': ['RES - Taxe sur la valeur ajoutée - Liquidation - Taux de TVA applicable aux systèmes de fixation permettant d’accrocher un fauteuil roulant à une trottinette électrique']}],
 'paragraphs_preview': [{'order_index': 0,
   'section_id': '12471-PGP__section__0000__res-taxe-sur-la-valeur-ajout-e-liquida

## 3. À quoi ressemble un document complet une fois transformé en chunks

Ici on regarde **tous** les chunks `section_window` d'un document réel.


In [8]:
def chunks_for_doc(chunks, boi_reference):
    return [chunk for chunk in chunks if chunk.boi_reference == boi_reference]

def chunk_view(chunks, boi_reference):
    rows = []
    for idx, chunk in enumerate(chunks_for_doc(chunks, boi_reference), start=1):
        rows.append({
            "idx": idx,
            "chunk_id": chunk.chunk_id,
            "chunk_kind": chunk.chunk_kind,
            "tokens": chunk.token_count,
            "section_path": " > ".join(chunk.section_path),
            "paragraph_range": chunk.paragraph_range,
            "text": chunk.text,
        })
    return rows

chunk_view(chunks_200_sw, "BOI-BIC-CHAMP-80-20-20-20-20240703")


[{'idx': 1,
  'chunk_id': '5357-PGP__section_window__5357-pgp-paragraph-00000__5357-pgp-paragraph-00005__bic-champ-d-application-et-territorialit-exon-rations-entreprises-exer-ant-une-activit-particuli-re-jeunes-entreprises-innovantes-jeunes-entreprises-universitaires-et-jeunes-entreprises-de-croissance-port-e-et-calcul-des-all-gements-fiscaux',
  'chunk_kind': 'paragraph_window',
  'tokens': 300,
  'section_path': "BIC - Champ d'application et territorialité - Exonérations - Entreprises exerçant une activité particulière - Jeunes entreprises innovantes, jeunes entreprises universitaires et jeunes entreprises de croissance - Portée et calcul des allègements fiscaux",
  'paragraph_range': ['5357-PGP__paragraph__00000',
   '5357-PGP__paragraph__00001',
   '5357-PGP__paragraph__00002',
   '5357-PGP__paragraph__00003',
   '5357-PGP__paragraph__00004',
   '5357-PGP__paragraph__00005'],
  'text': "Actualité liée : [node:date:14329-PGP] : BIC - IF - Suppression de l'exonération d'impôt sur le

In [9]:
chunk_view(chunks_200_sw, "BOI-IS-GPE-20-20-70-20190731")


[{'idx': 1,
  'chunk_id': '5035-PGP__section_window__5035-pgp-paragraph-00000__5035-pgp-paragraph-00002__is-r-gime-fiscal-des-groupes-de-soci-t-s-retraitements-n-cessaires-la-d-termination-du-r-sultat-et-de-la-plus-ou-moins-value-d-ensemble-traitement-du-m-canisme-de-la-sous-capitalisation',
  'chunk_kind': 'paragraph_window',
  'tokens': 234,
  'section_path': "IS - Régime fiscal des groupes de sociétés - Retraitements nécessaires à la détermination du résultat et de la plus ou moins-value d'ensemble - Traitement du mécanisme de la sous-capitalisation",
  'paragraph_range': ['5035-PGP__paragraph__00000',
   '5035-PGP__paragraph__00001',
   '5035-PGP__paragraph__00002'],
  'text': "L' article 34 de la loi n° 2018-1317 du 28 décembre 2018 de finances pour 2019 a abrogé les II et III de l' article 212 du code général des impôts (CGI) et modifié les dispositions de l' article 223 B du code général des impôts (CGI) pour les exercices ouverts à compter du 1 er janvier 2019. Les commentaires

## 4. Exemples réels de retrieval sur le subset `200`

On montre les sorties réelles:
- lexical
- dense
- hybrid pondéré


In [10]:
def report_rows(report):
    return {row["id"]: row for row in report["results"]}

rows_lex_200 = report_rows(reports["lex200"])
rows_dense_200 = report_rows(reports["dense200"])
rows_hyb_200 = report_rows(reports["hyb200_w"])

def chunk_snippet(chunk_map, chunk_id):
    chunk = chunk_map.get(chunk_id)
    if chunk is None:
        return None
    return {
        "boi_reference": chunk.boi_reference,
        "chunk_id": chunk.chunk_id,
        "chunk_kind": chunk.chunk_kind,
        "tokens": chunk.token_count,
        "section_path": " > ".join(chunk.section_path),
        "text": chunk.text[:320],
    }

def retrieval_trace_200(query_id):
    lex = rows_lex_200[query_id]
    dense = rows_dense_200[query_id]
    hyb = rows_hyb_200[query_id]
    return {
        "query_id": query_id,
        "query": lex["query"],
        "expected_boi": lex["expected_boi"],
        "lexical_top3": [chunk_snippet(chunk_by_id_200, hit["chunk_id"]) for hit in lex["top_hits"][:3]],
        "dense_top3": [chunk_snippet(chunk_by_id_200, hit["chunk_id"]) for hit in dense["top_hits"][:3]],
        "hybrid_top3_docs": hyb["top_hits"][:3],
    }

retrieval_trace_200("q01")


{'query_id': 'q01',
 'query': 'une JEI peut-elle demander le remboursement immédiat du CIR',
 'expected_boi': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
 'lexical_top3': [{'boi_reference': 'BOI-BIC-RICI-10-10-50-20250813',
   'chunk_id': '4669-PGP__section_window__4669-pgp-paragraph-00007__4669-pgp-paragraph-00014__ii-remboursement-imm-diat-des-cr-ances-de-cir-pour-certaines-entreprises',
   'chunk_kind': 'paragraph_window',
   'tokens': 339,
   'section_path': 'II. Remboursement immédiat des créances de CIR pour certaines entreprises',
   'text': 'En application des dispositions du I de l’ article 199 ter B du CGI , le CIR est imputé sur l’impôt sur les bénéfices dû par le contribuable au titre de l’année au cours de laquelle les dépenses de recherche prises en compte dans le calcul du crédit d’impôt ont été exposées. Les entreprises peuvent utiliser les créance'},
  {'boi_reference': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'chunk_id': '5357-PGP__section_window__5357-pgp-paragraph-00013__

In [11]:
retrieval_trace_200("q09")


{'query_id': 'q09',
 'query': 'traitement du mécanisme de la sous-capitalisation dans les groupes de sociétés',
 'expected_boi': 'BOI-IS-GPE-20-20-70-20190731',
 'lexical_top3': [{'boi_reference': 'BOI-IS-GPE-20-20-70-20190731',
   'chunk_id': '5035-PGP__section_window__5035-pgp-paragraph-00000__5035-pgp-paragraph-00002__is-r-gime-fiscal-des-groupes-de-soci-t-s-retraitements-n-cessaires-la-d-termination-du-r-sultat-et-de-la-plus-ou-moins-value-d-ensemble-traitement-du-m-canisme-de-la-sous-capitalisation',
   'chunk_kind': 'paragraph_window',
   'tokens': 234,
   'section_path': "IS - Régime fiscal des groupes de sociétés - Retraitements nécessaires à la détermination du résultat et de la plus ou moins-value d'ensemble - Traitement du mécanisme de la sous-capitalisation",
   'text': "L' article 34 de la loi n° 2018-1317 du 28 décembre 2018 de finances pour 2019 a abrogé les II et III de l' article 212 du code général des impôts (CGI) et modifié les dispositions de l' article 223 B du co

In [12]:
retrieval_trace_200("q12")


{'query_id': 'q12',
 'query': 'détermination du redevable de la TVA pour les livraisons de biens et prestations de services',
 'expected_boi': 'BOI-TVA-DECLA-10-10-20-20251022',
 'lexical_top3': [{'boi_reference': 'BOI-TVA-CHAMP-30-30-30-20190109',
   'chunk_id': '2267-PGP__section_window__2267-pgp-paragraph-00000__2267-pgp-paragraph-00004__tva-champ-d-application-et-territorialit-exon-rations-livraisons-et-prestations-de-services-portant-sur-les-bateaux-les-a-ronefs-et-leur-cargaison',
   'chunk_kind': 'paragraph_window',
   'tokens': 168,
   'section_path': "TVA - Champ d'application et territorialité - Exonérations - Livraisons et prestations de services portant sur les bateaux, les aéronefs et leur cargaison",
   'text': "Le II de l' article 262 du code général des impôts (CGI) exonère de la TVA : - les livraisons et les prestations de services portant sur les bateaux et les aéronefs ; - les livraisons et prestations de services portant sur les objets destinés à leur être incorporé

In [13]:
retrieval_trace_200("q22")


{'query_id': 'q22',
 'query': 'taux de TVA des cryptomonnaies minées par un particulier en 2026',
 'expected_boi': None,
 'lexical_top3': [{'boi_reference': 'BOI-TVA-SECT-70-30-20150902',
   'chunk_id': '691-PGP__section_window__691-pgp-paragraph-00000__691-pgp-paragraph-00004__tva-r-gimes-sectoriels-op-rations-intracommunautaires-portant-sur-les-moyens-de-transport-neufs-acquisition-intracommunautaire',
   'chunk_kind': 'paragraph_window',
   'tokens': 330,
   'section_path': 'TVA - Régimes sectoriels - Opérations intracommunautaires portant sur les moyens de transport neufs - Acquisition intracommunautaire',
   'text': "Toute personne résidant ou installée en France peut acquérir librement un moyen de transport neuf dans un autre État membre de la Communauté européenne. L'achat dans un autre État membre d'un moyen de transport neuf par une entreprise française redevable de la TVA obéit aux règles générales de taxation des acquisitions"},
  {'boi_reference': 'BOI-TVA-DECLA-10-10-20-20

## 5. Exemples réels de retrieval sur le subset `1000`

Même logique, mais avec beaucoup plus de distracteurs.


In [14]:
rows_lex_1000 = report_rows(reports["lex1000"])
rows_dense_1000 = report_rows(reports["dense1000"])
rows_hyb_1000 = report_rows(reports["hyb1000_w"])

def retrieval_trace_1000(query_id):
    lex = rows_lex_1000[query_id]
    dense = rows_dense_1000[query_id]
    hyb = rows_hyb_1000[query_id]
    return {
        "query_id": query_id,
        "query": lex["query"],
        "expected_boi": lex["expected_boi"],
        "lexical_top3": [chunk_snippet(chunk_by_id_1000, hit["chunk_id"]) for hit in lex["top_hits"][:3]],
        "dense_top3": [chunk_snippet(chunk_by_id_1000, hit["chunk_id"]) for hit in dense["top_hits"][:3]],
        "hybrid_top3_docs": hyb["top_hits"][:3],
    }

retrieval_trace_1000("q01")


{'query_id': 'q01',
 'query': 'une JEI peut-elle demander le remboursement immédiat du CIR',
 'expected_boi': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
 'lexical_top3': [{'boi_reference': 'BOI-BIC-RICI-10-10-50-20250813',
   'chunk_id': '4669-PGP__section_window__4669-pgp-paragraph-00007__4669-pgp-paragraph-00014__ii-remboursement-imm-diat-des-cr-ances-de-cir-pour-certaines-entreprises',
   'chunk_kind': 'paragraph_window',
   'tokens': 339,
   'section_path': 'II. Remboursement immédiat des créances de CIR pour certaines entreprises',
   'text': 'En application des dispositions du I de l’ article 199 ter B du CGI , le CIR est imputé sur l’impôt sur les bénéfices dû par le contribuable au titre de l’année au cours de laquelle les dépenses de recherche prises en compte dans le calcul du crédit d’impôt ont été exposées. Les entreprises peuvent utiliser les créance'},
  {'boi_reference': 'BOI-BIC-CHAMP-80-20-20-20-20240703',
   'chunk_id': '5357-PGP__section_window__5357-pgp-paragraph-00013__

In [15]:
retrieval_trace_1000("q12")


{'query_id': 'q12',
 'query': 'détermination du redevable de la TVA pour les livraisons de biens et prestations de services',
 'expected_boi': 'BOI-TVA-DECLA-10-10-20-20251022',
 'lexical_top3': [{'boi_reference': 'BOI-TVA-DECLA-10-10-20200323',
   'chunk_id': '3163-PGP__section_window__3163-pgp-paragraph-00000__3163-pgp-paragraph-00005__tva-r-gimes-d-imposition-et-obligations-d-claratives-et-comptables-redevable-de-la-taxe-livraisons-de-biens-et-prestations-de-services',
   'chunk_kind': 'paragraph_window',
   'tokens': 151,
   'section_path': "TVA - Régimes d'imposition et obligations déclaratives et comptables - Redevable de la taxe - Livraisons de biens et prestations de services",
   'text': "Le redevable de la taxe sur la valeur ajoutée (TVA) afférente à une livraison de biens ou une prestation est en principe l'assujetti qui réalise l'opération imposable ( code général des impôts (CGI), art. 283-1-al.1 ). Toutefois, l'acquéreur des biens ou le preneur des services peut être dési

In [16]:
retrieval_trace_1000("q22")


{'query_id': 'q22',
 'query': 'taux de TVA des cryptomonnaies minées par un particulier en 2026',
 'expected_boi': None,
 'lexical_top3': [{'boi_reference': 'BOI-TVA-DECLA-20-20-60-10-20220629',
   'chunk_id': '13240-PGP__section_window__13240-pgp-paragraph-00070__13240-pgp-paragraph-00073__iii-r-gime-particulier-ioss-a-assujettis-concern-s-par-le-r-gime-ioss',
   'chunk_kind': 'paragraph_window',
   'tokens': 314,
   'section_path': 'III. Régime particulier « IOSS » > A. Assujettis concernés par le régime « IOSS »',
   'text': "les assujettis établis dans un État non membre de l'Union européenne avec lequel la France dispose d'un instrument juridique relatif à l'assistance mutuelle ayant une portée similaire à celle prévue par la directive 2010/24/UE du Conseil du 16 mars 2010 concernant l'assistance mutuelle en matière de recouvrement des cr"},
  {'boi_reference': 'BOI-TVA-SECT-90-60-20250514',
   'chunk_id': '1292-PGP__section_window__1292-pgp-paragraph-00049__1292-pgp-paragraph-000

## 6. Conclusion actuelle, sans bullshit

Ce notebook montre l'état réel du clean-room:
- parsing: oui, robuste à grande échelle
- chunking baseline: oui, `section_window`
- retrieval lexical: bon baseline
- dense: encore trop confus à mesure qu'on augmente les distracteurs
- hybrid pondéré: utile, mais pas miracle
- abstention: pas encore implémentée

Donc:
- non, tout n'est pas “déjà bon”
- oui, on a maintenant un socle transparent et inspectable
- oui, on peut continuer sans se mentir sur ce qui fonctionne réellement
